# 39. Inference, temperature, top-p, and generation

**Level:** Intermediate to advanced

**Prerequisites:** 10, 37-38: probabilities and next-token models.

[Course index](00_START_HERE.ipynb) | [Previous](38_Language_Model_Pretraining_and_Next_Tokens.ipynb) | [Next](40_Prompts_Context_and_Model_Boundaries.ipynb)

**Topics:** Model inference, Temperature, Top-P, Top-k, Sampling, Autoregressive generation.

**Matching script:** `examples\39_Inference_Temperature_TopP_and_Generation.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

Autoregressive generation repeatedly computes a next-token distribution, selects a token,
appends it to the context, and stops at a condition such as EOS or a maximum output length.
Greedy decoding chooses the largest score; sampling draws from a distribution.

Temperature rescales logits before softmax: lower positive temperature concentrates mass,
higher temperature spreads it. Temperature zero is conventionally a service's greedy special
case, not ordinary division by zero. Even greedy hosted outputs may vary due to implementation,
batching, or model updates.

Top-k restricts candidates to the k highest-scoring tokens. Top-p retains the smallest ranked
prefix whose cumulative mass reaches p, then renormalizes. Include the boundary-crossing token;
dropping it is a common nucleus-sampling bug. Provider semantics and supported combinations vary.

Sampling controls diversity, not truth. Low temperature does not remove hallucination. Compare
outputs with task-level evaluations and enforce budgets for output length, latency, and cost.

## Mathematics or data contract

P_i = exp(logit_i / T) / sum_j exp(logit_j / T), for T > 0.

## Important limitation

The fixed logits are not from a pretrained model; this lesson isolates decoding mathematics.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Stable temperature scaling and correct nucleus selection

**Purpose:** Test the top-p boundary explicitly.

### Read the code in this order
1. Convert logits with a stable softmax.
2. Sort probabilities in descending order.
3. Keep the boundary token needed to reach p.
4. Sample with a seeded RNG and assert membership only in the retained set.

### Syntax and behavior to notice
- `assert` checks the stated behavior; a failed check raises AssertionError.
- `raise` reports invalid input or a failed requirement instead of fabricating success.
- A fixed random seed makes the chosen sampling reproducible within the environment.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
top-p retained: [0, 1] renormalized: [0.667, 0.333]
sampled token IDs: [1, 0, 1, 1, 0, 1, 1, 1, 0, 0]
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
import numpy as np

def distribution(logits, temperature):
    if temperature <= 0:
        raise ValueError("positive temperature required; use argmax for greedy")
    scaled = np.asarray(logits, dtype=float) / temperature
    values = np.exp(scaled - scaled.max())
    return values / values.sum()

def nucleus(probabilities, p):
    if not 0 < p <= 1:
        raise ValueError("p must be in (0, 1]")
    order = np.argsort(-probabilities)
    count = min(len(order), int(np.searchsorted(np.cumsum(probabilities[order]), p)) + 1)
    kept = order[:count]
    weights = probabilities[kept] / probabilities[kept].sum()
    return kept, weights

probabilities = np.array([0.6, 0.3, 0.1])
kept, weights = nucleus(probabilities, 0.7)
assert kept.tolist() == [0, 1]
assert distribution([2, 1, 0], 0.5).max() > distribution([2, 1, 0], 2).max()
samples = np.random.default_rng(42).choice(kept, size=10, p=weights)
assert set(samples).issubset({0, 1})
print("top-p retained:", kept.tolist(), "renormalized:", np.round(weights, 3).tolist())
print("sampled token IDs:", samples.tolist())

top-p retained: [0, 1] renormalized: [0.667, 0.333]
sampled token IDs: [1, 0, 1, 1, 0, 1, 1, 1, 0, 0]


### Interpret example 1

The result should demonstrate this purpose: **Test the top-p boundary explicitly.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Implement top-k, test k=1 against greedy selection, and reject invalid probability thresholds.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `Model inference` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: P_i = exp(logit_i / T) / sum_j exp(logit_j / T), for T > 0.
- Name the boundary of the demonstration: The fixed logits are not from a pretrained model; this lesson isolates decoding mathematics.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://docs.pytorch.org/tutorials/beginner/basics/intro.html`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.